# Problem Framing & Context

**Real-world problem (health & social impact):** Diabetes is a major chronic disease associated with preventable complications (e.g., cardiovascular disease, kidney disease) and high healthcare costs. Earlier identification of people likely to have diabetes can improve screening efficiency and help target prevention or clinical follow-up.

**ML framing (binary classification):**  
Given routinely collected health indicators and demographic characteristics, predict **whether an individual has diabetes**:

- **0 — Non-diabetes**
- **1 — Diabetes**

**Who is affected:**  
- Individuals who may benefit from earlier screening, diagnosis, and care pathways.  
- Clinicians and public health practitioners prioritising limited screening and prevention resources.  
- Health systems planning targeted interventions for high-risk populations.

**Decision this model could support:**  
Support **risk-based screening prioritisation** and **preventive outreach** (e.g., inviting higher-risk individuals for confirmatory testing, counselling, or follow-up), while keeping final decisions under human oversight.

### Hypothesis & Intended Impact
**Why this matters:**  
If we can reliably predict diabetes likelihood from accessible indicators, screening and follow-up resources can be targeted more effectively—potentially improving early detection and reducing downstream complications.

**Hypothesis (testable):**  
*Given health indicators (e.g., BMI, hypertension, cholesterol status, physical activity) and demographic variables, a supervised learning model can accurately classify whether an individual is diabetic (1) or non-diabetic (0), enabling more targeted screening and prevention efforts.*

**Intended impact (how predictions lead to action):**
- **Operational:** Help prioritise who should receive **screening invitations** or follow-up assessments first.  
- **Preventive:** Identify higher-risk individuals who may benefit from **lifestyle interventions** and monitoring.  
- **System-level:** Support **population health planning** by locating segments with elevated predicted risk.

**Ethical guardrails (intent):**
This model is **decision support**, not diagnosis. Any downstream use should include transparency, calibration checks, and subgroup performance evaluation to avoid harming underserved groups.


# Import Libraries

In [11]:
import pandas as pd
from pathlib import Path

# Dataset Discovery & Selection

data source: pip install ucimlrepo

**Selected dataset:** We are using a dataset from https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators. The Diabetes Health Indicators Dataset contains healthcare statistics and lifestyle survey information about people in general along with their diagnosis of diabetes. The 35 features consist of some demographics, lab test results, and answers to survey questions for each patient. The target variable for classification is whether a patient has diabetes, is pre-diabetic, or healthy.

**In this project:** the working label column is `Diabetes_binary` with values `{0, 1}`.

**Why this dataset fits the task:**
- Contains **risk-factor features** plausibly associated with diabetes (e.g., BMI, blood pressure, cholesterol, health behaviours).
- Suitable scale for training and comparing multiple classifiers.
- Includes demographic and access-to-care variables that enable explicit ethical review (fairness, representativeness).

**Access method (reproducible):**
- Dataset is loaded from the project’s `data/` folder or via a documented external source (depending on licensing/allowed redistribution).
- The notebook should include a single “source of truth” link and a deterministic loading step (e.g., fixed URL or dataset id) to ensure reproducibility.

**Licensing / legal note:**
Only store/redistribute the dataset in `data/` if the dataset license explicitly permits it. Otherwise, load it programmatically and document the source and retrieval steps in the notebook.


In [12]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cdc_diabetes_health_indicators = fetch_ucirepo(id=891) 
  
# data (as pandas dataframes) 
X = cdc_diabetes_health_indicators.data.features 
y = cdc_diabetes_health_indicators.data.targets 
  
# metadata 
print(cdc_diabetes_health_indicators.metadata) 
  
# variable information 
print(cdc_diabetes_health_indicators.variables) 


{'uci_id': 891, 'name': 'CDC Diabetes Health Indicators', 'repository_url': 'https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators', 'data_url': 'https://archive.ics.uci.edu/static/public/891/data.csv', 'abstract': 'The Diabetes Health Indicators Dataset contains healthcare statistics and lifestyle survey information about people in general along with their diagnosis of diabetes. The 35 features consist of some demographics, lab test results, and answers to survey questions for each patient. The target variable for classification is whether a patient has diabetes, is pre-diabetic, or healthy. ', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Tabular', 'Multivariate'], 'num_instances': 253680, 'num_features': 21, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Sex', 'Age', 'Education Level', 'Income'], 'target_col': ['Diabetes_binary'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_

In [13]:
df_X = pd.DataFrame(X, columns=cdc_diabetes_health_indicators.metadata.feature_names)
df_X.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,1,1,1,40,1,0,0,0,0,1,...,1,0,5,18,15,1,0,9,4,3
1,0,0,0,25,1,0,0,1,0,0,...,0,1,3,0,0,0,0,7,6,1
2,1,1,1,28,0,0,0,0,1,0,...,1,1,5,30,30,1,0,9,4,8
3,1,0,1,27,0,0,0,1,1,1,...,1,0,2,0,0,0,0,11,3,6
4,1,1,1,24,0,0,0,1,1,1,...,1,0,2,3,0,0,0,11,5,4


In [14]:
df_y = pd.DataFrame(y, columns=cdc_diabetes_health_indicators.metadata.target_names)
df_y.head()

,Diabetes_binary
0,0
1,0
2,0
3,0
4,0


In [15]:
# check if really binary
{col: df_y[col].unique() for col in df_y.columns}


{'Diabetes_binary': array([0, 1])}

### Why this dataset?
This dataset is appropriate because it pairs a diabetes outcome label with a set of **measurable health indicators** and **behavioural/demographic factors** that are meaningful for screening and prevention workflows. Many of the inputs are modifiable (e.g., physical activity), which makes the results more actionable than purely genetic or highly technical clinical signals.

### Target Variable (label)
**Target:** `Diabetes_binary`  
- **0:** non-diabetes  
- **1:** diabetes  

**Why this target is meaningful:**
- Directly aligns with a screening-support use case: “who is more likely to have diabetes?”
- Actionable for stakeholders: higher-risk predictions can trigger **confirmatory testing** or additional clinical evaluation.

**Feasibility and caution:**
- Feasible because established correlates of diabetes risk exist in the feature set.
- Requires careful evaluation due to possible **class imbalance**, **self-report noise**, and **access-to-care effects** that may influence the label.

### Features Overview (and dataset documentation)
This project uses the **CDC Diabetes Health Indicators** dataset as hosted by the :contentReference[oaicite:0]{index=0}. The features below come from a public-health survey pipeline associated with the :contentReference[oaicite:1]{index=1} and the :contentReference[oaicite:2]{index=2} (per the dataset overview).

| Feature | Type | What it represents | How it may help prediction | Early concerns to revisit |
|---|---|---|---|---|
| `HighBP` | Binary | Ever told you have high blood pressure | Metabolic/cardiovascular comorbidity correlated with diabetes | May reflect healthcare access/diagnosis |
| `HighChol` | Binary | Ever told you have high cholesterol | Metabolic risk marker | Healthcare access/diagnosis bias |
| `CholCheck` | Binary | Cholesterol checked within last 5 years | Preventive care utilisation | **Proxy for access to care**; can shift label visibility |
| `BMI` | Numerical | Body Mass Index | Strong physiological correlate of diabetes | Nonlinear effects; outliers |
| `Smoker` | Binary | Smoked ≥100 cigarettes in lifetime (typical coding) | Lifestyle risk factor | Self-report noise; social desirability bias |
| `Stroke` | Binary | History of stroke | Severe comorbidity often linked to chronic disease burden | May capture downstream effects rather than causes |
| `HeartDiseaseorAttack` | Binary | Coronary heart disease or myocardial infarction | Comorbidity linked to metabolic syndrome | Same as above; may encode severity/age |
| `PhysActivity` | Binary | Physical activity indicator | Modifiable protective/risk factor | Self-report noise |
| `Fruits` | Binary | Fruit consumption indicator | Proxy for diet quality | Coarse measure; cultural/SES confounding |
| `Veggies` | Binary | Vegetable consumption indicator | Proxy for diet quality | Coarse measure; cultural/SES confounding |
| `HvyAlcoholConsump` | Binary | Heavy alcohol consumption indicator | Lifestyle/health behaviour signal | Self-report; potential subgroup bias |
| `AnyHealthcare` | Binary | Has any kind of health coverage | Access to healthcare | **Strong access proxy**; can affect diagnosis likelihood |
| `NoDocbcCost` | Binary | Could not see doctor due to cost | Financial barrier to care | **SES/access proxy**; fairness risk |
| `GenHlth` | Ordinal categorical (often 1–5) | Self-rated general health | Captures overall health burden | Subjective reporting differences across groups |
| `MentHlth` | Numerical (days) | Poor mental health days (past 30) | Stress/health burden association | Subjective reporting; may correlate with SES |
| `PhysHlth` | Numerical (days) | Poor physical health days (past 30) | Health burden / chronic illness load | Subjective reporting; may reflect complications |
| `DiffWalk` | Binary | Serious difficulty walking/climbing stairs | Functional limitation | Could encode disability-related disparities |
| `Sex` | Binary | Sex | Baseline risk stratification | **Sensitive attribute** → evaluate subgroup performance |
| `Age` | Ordinal categorical (binned) | Age category | Strong baseline predictor | **Sensitive attribute**; check calibration by age |
| `Education` | Ordinal categorical | Education attainment | Social determinant correlated with risk and care | **SES proxy**; fairness risk |
| `Income` | Ordinal categorical | Household income category | Social determinant correlated with risk and care | **SES proxy**; fairness risk |

### Notes to carry into data prep & evaluation
- **Access-to-care features (`CholCheck`, `AnyHealthcare`, `NoDocbcCost`)** may improve accuracy but can also introduce *diagnosis visibility bias* (people with better access are more likely to be diagnosed). Consider: (a) fairness metrics by SES proxies, (b) sensitivity analysis with/without access variables.
- **Class imbalance** is common in diabetes datasets. Use stratified splits and report **balanced accuracy**, **F1**, and **PR-AUC** (especially if positives are rare).
- **Sensitive/proxy attributes** (`Sex`, `Age`, `Education`, `Income`) should trigger explicit subgroup evaluation (e.g., per-group recall/false negative rate), since screening-related errors can cause unequal harm.

# Save dataset

In [16]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# combine features and target into one dataframe
cleaned_df = df_X.copy()
for col in df_y.columns:
    cleaned_df[col] = df_y[col].values

# save to CSV
output_path = processed_dir / "original_data.csv"
cleaned_df.to_csv(output_path, index=False)

print("Saved:", output_path.resolve())
print("Shape:", cleaned_df.shape)
cleaned_df.head()


Saved: C:\Users\Lu\OneDrive\ToU\chl_Classification\data\processed\original_data.csv
Shape: (253680, 22)


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0
